# %% [markdown]
# # NHANES: Weighted Ensemble Clustering — Full Results Pipeline
# This notebook loads the local NHANES merges, prepares data for clustering,
# runs the weighted ensemble + consensus pipeline, and produces all figures & tables
# for the Results section (eigengap plot, PCA plots, weighted barplots, regression tables & forest plot).


In [ ]:
# %%
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, SpectralClustering, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances_argmin
from sklearn.metrics import adjusted_rand_score
from scipy.sparse import coo_matrix, diags
from scipy.sparse.linalg import eigsh
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score


sns.set(style="whitegrid", context="talk")
np.random.seed(42)

In [ ]:
# ============================================================
# Configure output directories for generated results and figures
# ============================================================

from pathlib import Path

# Base output directory
OUT_DIR = Path("NHANES_OUT")

# Directory for publication figures
FIG_DIR = OUT_DIR / "figures"

# Create directories if they do not already exist
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# %% [markdown]
# ## 1) Import NHANES-like Dataset

In [ ]:
# %%
DATA_DIR = r"C:\Users\afees\NHANES_OUT"   # <-- change only if needed
COMBINED = os.path.join(DATA_DIR, "merged_2015_2018.csv")

# If you don't have the combined file, load per-cycle and concat:
if os.path.exists(COMBINED):
    df = pd.read_csv(COMBINED)
else:
    f1 = os.path.join(DATA_DIR, "merged_2015_2016.csv")
    f2 = os.path.join(DATA_DIR, "merged_2017_2018.csv")
    df1 = pd.read_csv(f1) if os.path.exists(f1) else pd.DataFrame()
    df2 = pd.read_csv(f2) if os.path.exists(f2) else pd.DataFrame()
    cols = sorted(set(df1.columns) | set(df2.columns))
    df1 = df1.reindex(columns=cols)
    df2 = df2.reindex(columns=cols)
    df = pd.concat([df1, df2], ignore_index=True)

# Adults only
# Adults only – robust handling if RIDAGEYR is missing or renamed
age_col_candidates = [c for c in df.columns if c.upper().startswith("RIDAGEYR")]

if len(age_col_candidates) == 0:
    raise KeyError(
        f"No age column like 'RIDAGEYR' found. Columns available include:\n{df.columns.tolist()}"
    )

age_col = age_col_candidates[0]  # e.g. 'RIDAGEYR' or 'RIDAGEYR_x'
print(f"Using age column: {age_col}")

df = df[df[age_col] >= 18].copy()


In [ ]:
import os

print("Combined exists:", os.path.exists(COMBINED))
print("2015-2016 exists:", os.path.exists(os.path.join(DATA_DIR, "merged_2015_2016.csv")))
print("2017-2018 exists:", os.path.exists(os.path.join(DATA_DIR, "merged_2017_2018.csv")))

print("\nFiles in DATA_DIR:")
print(os.listdir(DATA_DIR))


In [ ]:
# Build analysis weight (prefer fasting weight when fasting labs present)
has_fast = pd.Series(False, index=df.index)
if "LBXGLU" in df.columns:
    has_fast |= df["LBXGLU"].notna()
if "LBXTR" in df.columns:
    has_fast |= df["LBXTR"].notna()

df["W_ANALYSIS"] = df.get("WTMEC2YR", np.nan)
if "WTSAF2YR" in df.columns:
    df.loc[has_fast & df["WTSAF2YR"].notna(), "W_ANALYSIS"] = df.loc[has_fast, "WTSAF2YR"]

# Weights strictly positive
df["W_ANALYSIS"] = df["W_ANALYSIS"].clip(lower=0.001)

In [ ]:
# Fallback BP means if not present
def _mean_across(cols):
    c = [c for c in cols if c in df.columns]
    return df[c].mean(axis=1) if c else pd.Series(np.nan, index=df.index)

if "SBP_mean" not in df.columns:
    df["SBP_mean"] = _mean_across(["BPXSY1", "BPXSY2", "BPXSY3"])
if "DBP_mean" not in df.columns:
    df["DBP_mean"] = _mean_across(["BPXDI1", "BPXDI2", "BPXDI3"])

In [ ]:
# Map NHANES -> analysis columns
sex_map = {1: "Male", 2: "Female"}
cols_map = {
    "Patient_ID":         "SEQN",
    "Age":                "RIDAGEYR",
    "Sex":                "RIAGENDR",
    "BMI":                "BMXBMI",
    "Waist_Circumference":"BMXWAIST",
    "Systolic_BP":        "SBP_mean",
    "Diastolic_BP":       "DBP_mean",
    "HbA1c":              "LBXGH",
    "Fasting_Glucose":    "LBXGLU",
    "Triglycerides":      "LBXTR",
    "HDL_Cholesterol":    "LBDHDD",
    "LDL_Cholesterol":    "LBDLDL",   # may be missing
    "Survey_Weight":      "W_ANALYSIS",
}
present = {k: v for k, v in cols_map.items() if v in df.columns}
data = df.rename(columns={v: k for k, v in present.items()})[list(present.keys())].copy()

if "Sex" in data.columns:
    data["Sex"] = data["Sex"].map(sex_map).astype("category")

In [ ]:
# Ensure LDL exists (stub if missing)
if "LDL_Cholesterol" not in data.columns:
    data["LDL_Cholesterol"] = np.nan

# Keep those with at least some clinical/lab data
at_least_one = ["HbA1c","Fasting_Glucose","BMI","Waist_Circumference",
                "Systolic_BP","Diastolic_BP","Triglycerides","HDL_Cholesterol","LDL_Cholesterol"]
data = data.dropna(subset=at_least_one, how="all")

print("Data shape:", data.shape)
print(data.head(3))

# %% [markdown]
# ## 2) Quick Missingness and Distributions

In [ ]:
# %%
print("\n[Missing counts]")
print(data.isna().sum())
print("\n[Describe]")
print(data.describe(include="all"))

# %% [markdown]
# ## 3) Prep for Clustering (clip → transform → impute → scale)

In [ ]:
# %%
from sklearn.impute import SimpleImputer

def prepare_for_clustering(
    data: pd.DataFrame,
    require_fasting_panel=False,
    log_transform_triglycerides=True,
    clip_extremes=True,
    verbose=True
):
    df = data.copy()

    expected_cols = [
        "Patient_ID", "Age", "Sex", "BMI", "Waist_Circumference",
        "Systolic_BP", "Diastolic_BP", "HbA1c", "Fasting_Glucose",
        "Triglycerides", "HDL_Cholesterol", "LDL_Cholesterol", "Survey_Weight"
    ]
    missing_cols = [c for c in expected_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

    # Encode Sex (Female=0, Male=1)
    sex_map = {"Female": 0, "Male": 1, 0: 0, 1: 1, 2: 0}
    df["Sex"] = df["Sex"].map(sex_map)

    # Clip to plausible ranges
    if clip_extremes:
        clip_rules = {
            "Age": (18, 90),
            "BMI": (12, 80),
            "Waist_Circumference": (50, 200),
            "Systolic_BP": (70, 260),
            "Diastolic_BP": (30, 150),
            "HbA1c": (4, 15),
            "Fasting_Glucose": (50, 400),
            "Triglycerides": (20, 800),
            "HDL_Cholesterol": (10, 120),
            "LDL_Cholesterol": (20, 300),
        }
        for col, (lo, hi) in clip_rules.items():
            if col in df.columns:
                df[col] = df[col].clip(lower=lo, upper=hi)

    # Optional log1p for TG
    if log_transform_triglycerides and "Triglycerides" in df.columns:
        df["Triglycerides"] = np.where(
            df["Triglycerides"].gt(0),
            np.log1p(df["Triglycerides"]),
            np.nan
        )

    # Features (match pipeline)
    features = [
        "Age", "Sex", "BMI", "Waist_Circumference",
        "Systolic_BP", "Diastolic_BP",
        "HbA1c", "Fasting_Glucose",
        "Triglycerides", "HDL_Cholesterol", "LDL_Cholesterol"
    ]

    if require_fasting_panel:
        mask = df["Fasting_Glucose"].notna() & df["Triglycerides"].notna()
        df = df.loc[mask].copy()
        if verbose:
            print(f"[Cohort] Requiring fasting panel: kept {mask.sum()} rows.")

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    imputer = SimpleImputer(strategy="median")
    X_raw = imputer.fit_transform(df[features])

    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)

    weights = df["Survey_Weight"].to_numpy().astype(float)
    weights[np.isnan(weights) | (weights <= 0)] = 0.001

    cleaned_df = df[["Patient_ID"] + features + ["Survey_Weight"]].copy()
    if log_transform_triglycerides:
        cleaned_df.rename(columns={"Triglycerides": "Triglycerides_log1p"}, inplace=True)

    if verbose:
        miss = df[features].isna().sum().sort_values(ascending=False)
        print("\n[Missing after clipping, before imputation]")
        print(miss.to_string())
        print(f"\nFinal cohort size (rows): {df.shape[0]}")
        print(f"Features used ({len(features)}): {features}")

    return X, features, weights, cleaned_df

X, features_used, weights, cleaned_df = prepare_for_clustering(
    data,
    require_fasting_panel=False,
    log_transform_triglycerides=True,
    clip_extremes=True,
    verbose=True
)
print("\nX shape:", X.shape, "| Weights shape:", weights.shape)


# %% [markdown]
# ## 4) Ensemble: Base Clusterings → Weighted Co-association → Consensus

In [ ]:
# %%
n = X.shape[0]
w = np.asarray(weights, dtype=float)
w[w <= 0] = 0.001
w_bar2 = (w.mean() ** 2)

def weighted_kmeans(X, weights, n_clusters=3, max_iter=100, tol=1e-4, seed=42):
    rng = np.random.default_rng(seed)
    centroids = X[rng.choice(X.shape[0], n_clusters, replace=False)]
    for _ in range(max_iter):
        labels = pairwise_distances_argmin(X, centroids)
        new_centroids = np.zeros_like(centroids)
        for k in range(n_clusters):
            mask = labels == k
            if mask.sum() > 0:
                new_centroids[k] = np.average(X[mask], axis=0, weights=weights[mask])
        if np.all(np.linalg.norm(new_centroids - centroids, axis=1) < tol):
            break
        centroids = new_centroids
    return labels

In [ ]:
k_values = [3, 4, 5, 6]
seeds    = [0, 1]
methods  = ["kmeans", "wkmeans", "gmm", "spectral", "agglo"]

def run_base_clustering(X, method, k, seed, weights=None):
    if method == "kmeans":
        km = KMeans(n_clusters=k, n_init=10, random_state=seed)
        return km.fit_predict(X)
    elif method == "wkmeans":
        if weights is None:
            raise ValueError("weights required for wkmeans")
        return weighted_kmeans(X, weights, n_clusters=k, seed=seed)
    elif method == "gmm":
        gm = GaussianMixture(n_components=k, covariance_type="full",
                             random_state=seed, n_init=1)
        return gm.fit(X).predict(X)
    elif method == "spectral":
        sc = SpectralClustering(n_clusters=k, affinity="nearest_neighbors",
                                n_neighbors=25, assign_labels="kmeans",
                                random_state=seed, n_init=10)
        return sc.fit_predict(X)
    elif method == "agglo":
        ac = AgglomerativeClustering(n_clusters=k, linkage="ward")
        return ac.fit_predict(X)
    else:
        raise ValueError(method)

base_labels = []
from collections import Counter
meta_counts = Counter()
for method in methods:
    for k in k_values:
        for s in seeds:
            try:
                labels = run_base_clustering(X, method, k, s, weights=w)
                base_labels.append((method, k, s, labels))
                meta_counts[(method, k)] += 1
            except Exception as e:
                print(f"[WARN] {method} k={k} seed={s} failed: {e}")

B = len(base_labels)
print(f"[INFO] Total base partitions: {B} (includes weighted & unweighted K-means)")
print("[INFO] Partitions by (method,k):")
for (m,k), cnt in sorted(meta_counts.items()):
    print(f"  - {m:9s} k={k}: {cnt} runs")

In [ ]:
# kNN neighbor graph for sparsity
nn_k = 50
print(f"[INFO] Building kNN graph with k={nn_k} ...")
nbrs = NearestNeighbors(n_neighbors=nn_k+1, metric="euclidean").fit(X)
knn_idx = nbrs.kneighbors(return_distance=False)[:, 1:]

# Weighted co-association on neighbors
rows, cols, data_s = [], [], []
for (method, k, s, labels) in base_labels:
    for i in range(n):
        li = labels[i]
        js = knn_idx[i]
        same = js[labels[js] == li]
        if same.size:
            rows.extend([i] * same.size)
            cols.extend(same.tolist())
            data_s.extend([1.0] * same.size)

C_counts = coo_matrix((data_s, (rows, cols)), shape=(n, n)).tocsr()
C_counts = C_counts.maximum(C_counts.T)

C_freq = C_counts.copy().astype(np.float64)
C_freq.data = C_freq.data / float(B)

row_idx, col_idx = C_freq.nonzero()
pair_scale = (w[row_idx] * w[col_idx]) / w_bar2
C_freq.data *= pair_scale

I = diags(np.ones(n))
A = (C_freq + I).tocsr()
print(f"[INFO] Co-association nnz: {A.nnz} (sparse density ~ {A.nnz/(n*n):.6f})")

In [ ]:
# Eigengap selection
def select_k_by_eigengap(A, k_max=10):
    d = np.asarray(A.sum(axis=1)).ravel()
    d[d <= 1e-12] = 1e-12
    D_m12 = diags(d ** -0.5)
    L = (I - D_m12 @ A @ D_m12).tocsr()
    vals, _ = eigsh(L, k=min(k_max+1, n-2), which="SM")
    vals = np.sort(vals)
    gaps = np.diff(vals)
    k_star = int(np.argmax(gaps[:k_max-1]) + 1)
    return k_star, vals, gaps

k_star, lambdas, gaps = select_k_by_eigengap(A, k_max=10)
print(f"[INFO] Consensus K (eigengap): {k_star}")
print("[INFO] Smallest eigenvalues:", np.round(lambdas[:min(8, len(lambdas))], 5))
print("[INFO] First eigengaps:      ", np.round(gaps[:min(7, len(gaps))], 5))

consensus_labels_weighted = SpectralClustering(
    n_clusters=k_star,
    affinity="precomputed",
    assign_labels="kmeans",
    n_init=20,
    random_state=42
).fit_predict(A)


In [ ]:
# Unweighted consensus (for ARI)
C_freq_unw = C_counts.copy().astype(np.float64)
C_freq_unw.data = C_freq_unw.data / float(B)
A_unw = (C_freq_unw + I).tocsr()
k_star_unw, lambdas_unw, gaps_unw = select_k_by_eigengap(A_unw, k_max=10)
consensus_labels_unweighted = SpectralClustering(
    n_clusters=k_star_unw,
    affinity="precomputed",
    assign_labels="kmeans",
    n_init=20,
    random_state=42
).fit_predict(A_unw)

if k_star_unw == k_star:
    ari = adjusted_rand_score(consensus_labels_unweighted, consensus_labels_weighted)
    print(f"[RESULT] ARI (weighted vs unweighted): {ari:.3f}")
else:
    print("[NOTE] ARI skipped (different K for weighted vs unweighted).")

# Attach and save
out = cleaned_df.copy()
out["Consensus_Cluster"] = consensus_labels_weighted
out.to_csv(os.path.join(OUT_DIR, "phenotypes_consensus_weighted.csv"), index=False)
np.save(os.path.join(OUT_DIR, "consensus_labels_weighted.npy"), consensus_labels_weighted)
np.save(os.path.join(OUT_DIR, "consensus_labels_unweighted.npy"), consensus_labels_unweighted)

In [ ]:
# %% [markdown]
# ## 5) Survey-weighted Prevalence & Cluster Summaries

In [ ]:
# %%
def weighted_prevalence(labels, weights):
    dfp = pd.DataFrame({"lab": labels, "w": weights})
    prev = dfp.groupby("lab", as_index=False)["w"].sum()
    prev["weighted_share"] = prev["w"] / prev["w"].sum()
    return prev.sort_values("lab").reset_index(drop=True)

print("\n[RESULT] Weighted prevalence by cluster:")
prev_table = weighted_prevalence(consensus_labels_weighted, weights)
print(prev_table.to_string(index=False))

def weighted_cluster_summary(df, cluster_col="Consensus_Cluster", weight_col="Survey_Weight", feats=None):
    feats = feats or [c for c in df.columns if c not in ("Patient_ID", "Survey_Weight", cluster_col)]
    rows = []
    for k in sorted(df[cluster_col].unique()):
        g = df[df[cluster_col] == k]
        wts = g[weight_col].astype(float).values
        row = {"Cluster": int(k), "N_unweighted": len(g), "W_sum": float(np.nansum(wts))}
        for f in feats:
            x = g[f].astype(float).values
            msk = ~np.isnan(x)
            if msk.any():
                m = np.average(x[msk], weights=wts[msk])
                v = np.average((x[msk] - m) ** 2, weights=wts[msk])
                row[f"{f}_mean"] = float(m)
                row[f"{f}_sd"]   = float(np.sqrt(v))
            else:
                row[f"{f}_mean"] = np.nan
                row[f"{f}_sd"]   = np.nan
        rows.append(row)
    return pd.DataFrame(rows).sort_values("Cluster").reset_index(drop=True)

summary_table = weighted_cluster_summary(
    out,
    feats=["Age","BMI","Waist_Circumference","Systolic_BP","Diastolic_BP",
           "HbA1c","Fasting_Glucose","HDL_Cholesterol","LDL_Cholesterol"]
)
summary_table.to_csv(os.path.join(OUT_DIR, "cluster_summary_weighted.csv"), index=False)
print("\n[RESULT] Survey-weighted cluster summary (head):")
print(summary_table.head().to_string(index=False))


# %% [markdown]
# ## 6) FIGURE — Eigengap Curve (Weighted Consensus Graph)


In [ ]:
# ============================================================
# Publication-quality eigengap curve
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import os
import seaborn as sns

# ------------------------------------------------------------
# X-axis values
# ------------------------------------------------------------
x_vals = np.arange(1, len(gaps) + 1)

# Optimal K from eigengap
optimal_k = np.argmax(gaps) + 1

# ------------------------------------------------------------
# Plot style
# ------------------------------------------------------------
sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(8.2, 6))

# Main eigengap curve
ax.plot(
    x_vals,
    gaps,
    marker="o",
    markersize=7,
    linewidth=2.2,
    color="#4C72B0"
)

# Highlight optimal eigengap
ax.scatter(
    optimal_k,
    gaps[optimal_k - 1],
    s=120,
    color="#DD8452",
    zorder=5,
    label=fr"Selected $K={optimal_k}$"
)

# Vertical reference line
ax.axvline(
    optimal_k,
    linestyle="--",
    linewidth=1.4,
    alpha=0.7
)

# Annotation
ax.annotate(
    fr"Largest eigengap at $K={optimal_k}$",
    xy=(optimal_k, gaps[optimal_k - 1]),
    xytext=(optimal_k + 0.4, gaps[optimal_k - 1] * 0.82),
    fontsize=11,
    arrowprops=dict(
        arrowstyle="->",
        lw=1.2
    )
)

# ------------------------------------------------------------
# Labels and title
# ------------------------------------------------------------
ax.set_xlabel(
    r"Cluster index $K$",
    fontsize=12
)

ax.set_ylabel(
    r"Eigengap $(\lambda_{i+1} - \lambda_i)$",
    fontsize=12
)

ax.set_title(
    "Eigengap Curve for the Weighted Consensus Graph",
    fontsize=14,
    pad=12
)

# ------------------------------------------------------------
# Ticks and limits
# ------------------------------------------------------------
ax.set_xticks(x_vals)
ax.set_xlim(1, len(gaps))

# ------------------------------------------------------------
# Grid and spines
# ------------------------------------------------------------
ax.grid(alpha=0.25)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Legend
ax.legend(frameon=True)

plt.tight_layout()

# ------------------------------------------------------------
# Save figures
# ------------------------------------------------------------
plt.savefig(
    os.path.join(FIG_DIR, "eigengap_weighted_improved.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(FIG_DIR, "eigengap_weighted_improved.pdf"),
    bbox_inches="tight"
)

plt.show()

In [ ]:
# === Weighted vs Unweighted Consensus Clustering, ARI & NMI, and Output Tables ===

print(">>> Building unweighted affinity matrix...")

C_freq_unw = C_counts.copy().astype(np.float64)
C_freq_unw.data = C_freq_unw.data / float(B)      # divide by number of base partitions
A_unw = (C_freq_unw + I).tocsr()

print(">>> Selecting k* for UNWEIGHTED consensus via eigengap...")
k_star_unw, lambdas_unw, gaps_unw = select_k_by_eigengap(A_unw, k_max=10)
print(f"    k*_unweighted = {k_star_unw}")

print(">>> Running spectral clustering for UNWEIGHTED consensus...")
consensus_labels_unweighted = SpectralClustering(
    n_clusters=k_star_unw,
    affinity="precomputed",
    assign_labels="kmeans",
    n_init=20,
    random_state=42
).fit_predict(A_unw)

print(f"    k*_weighted   = {k_star}")

# ---------------------------
# Agreement between weighted and unweighted consensus
# ---------------------------
print(">>> Computing agreement between weighted and unweighted consensus...")

ari = adjusted_rand_score(consensus_labels_unweighted, consensus_labels_weighted)
nmi = normalized_mutual_info_score(consensus_labels_unweighted,
                                   consensus_labels_weighted)

print(f"[RESULT] Adjusted Rand Index (weighted vs unweighted): {ari:.3f}")
print(f"[RESULT] Normalized Mutual Information (weighted vs unweighted): {nmi:.3f}")

agreement_metrics = {
    "ARI_weighted_vs_unweighted": float(ari),
    "NMI_weighted_vs_unweighted": float(nmi),
    "k_star_weighted": int(k_star),
    "k_star_unweighted": int(k_star_unw),
}

import json
os.makedirs(OUT_DIR, exist_ok=True)
with open(os.path.join(OUT_DIR, "consensus_agreement_metrics.json"), "w") as f:
    json.dump(agreement_metrics, f, indent=2)

# ---------------------------
# Attach BOTH sets of labels
# ---------------------------
print(">>> Attaching weighted and unweighted labels to cleaned_df...")

out = cleaned_df.copy()
out["cluster_weighted"]   = consensus_labels_weighted
out["cluster_unweighted"] = consensus_labels_unweighted

combined_csv_path = os.path.join(OUT_DIR, "phenotypes_consensus_weighted_unweighted.csv")
print(f">>> Saving combined cluster table to: {combined_csv_path}")
out.to_csv(combined_csv_path, index=False)

np.save(os.path.join(OUT_DIR, "consensus_labels_weighted.npy"),   consensus_labels_weighted)
np.save(os.path.join(OUT_DIR, "consensus_labels_unweighted.npy"), consensus_labels_unweighted)

print(">>> Done. You can now report ARI and NMI in the Results section.")
print(f"    ARI = {ari:.3f}, NMI = {nmi:.3f}")


# %% [markdown]
# ## 7) FIGURE — PCA Projection Colored by Consensus Cluster

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os

# -----------------------------
# PCA projection
# -----------------------------
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

# Create plotting dataframe
pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "Cluster": out["cluster_weighted"]
})

# Replace numeric labels with phenotype names
cluster_map = {
    0: "Metabolically Favorable",
    1: "Cardiometabolic Risk"
}

pca_df["Phenotype"] = pca_df["Cluster"].map(cluster_map)

# -----------------------------
# Plot
# -----------------------------
plt.figure(figsize=(7.5, 6.5))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="Phenotype",
    palette=["#4C72B0", "#DD8452"],
    s=12,                 # smaller points
    alpha=0.35,           # improved transparency
    linewidth=0,
    rasterized=True       # improves rendering for dense plots
)

plt.title(
    "PCA Projection of Survey-Weighted Consensus Phenotypes",
    fontsize=14,
    pad=12
)

plt.xlabel(
    f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance explained)",
    fontsize=12
)

plt.ylabel(
    f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance explained)",
    fontsize=12
)

plt.legend(
    title="Phenotype",
    frameon=True,
    fontsize=10,
    title_fontsize=11,
    loc="upper left"
)

plt.grid(alpha=0.15)
sns.despine()

plt.tight_layout()

# Save high-quality versions
plt.savefig(
    os.path.join(FIG_DIR, "pca_consensus_improved.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(FIG_DIR, "pca_consensus_improved.pdf"),
    bbox_inches="tight"
)

plt.show()

In [ ]:
from sklearn.decomposition import PCA

# Make sure X and out refer to the same rows (same length & order)
# X = cleaned_df[feature_cols].to_numpy()   # or however you built X

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(7, 6))
sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=out["cluster_weighted"].astype(str),  # <- use existing column name
    s=20,
    alpha=0.7,
    edgecolor=None,
    legend=True
)
plt.title("PCA Projection — Weighted Consensus Clusters")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "pca_consensus.png"), dpi=300)
plt.show()


# %% [markdown]
# ## 8) FIGURE — PCA Small Multiples for Base Methods (k=4)
# Shows diversity across base partitions for one k (adjust as needed).

In [ ]:
# %%
# Re-run base labels for k=4 only to plot side-by-side (uses the ones we already have too)
subset = [(m,k,s,lbl) for (m,k,s,lbl) in base_labels if k==4]
if len(subset) == 0:
    subset = []
    for m in ["kmeans","wkmeans","gmm","spectral","agglo"]:
        try:
            lbl = run_base_clustering(X, m, 4, seed=0, weights=w)
            subset.append((m,4,0,lbl))
        except Exception as e:
            print(f"[WARN] PCA small-multiples skipped for {m}: {e}")

nplots = min(4, len(subset))
if nplots > 0:
    fig, axes = plt.subplots(1, nplots, figsize=(4*nplots+2, 4), sharex=True, sharey=True)
    if nplots == 1:
        axes = [axes]
    for ax, (m,k,s,lbl) in zip(axes, subset[:nplots]):
        sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=lbl.astype(str),
                        s=12, alpha=0.7, edgecolor=None, legend=False, ax=ax)
        ax.set_title(f"PCA — {m} (k={k})")
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "pca_base_methods_k4.png"), dpi=300)
    plt.show()
else:
    print("[NOTE] No base partitions to plot for k=4.")

# %% [markdown]
# ## 9) FIGURE — Weighted Cluster Prevalence Bar Chart


In [ ]:
# ============================================================
# Publication-quality Figure:
# Population-weighted phenotype prevalence
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# ------------------------------------------------------------
# Prepare plotting dataframe
# ------------------------------------------------------------
plot_prev = prev_table.copy()

phenotype_map = {
    0: "Metabolically\nFavorable",
    1: "Cardiometabolic\nRisk"
}

plot_prev["Phenotype"] = plot_prev["lab"].map(phenotype_map)
plot_prev["Weighted_Percent"] = plot_prev["weighted_share"] * 100

# ------------------------------------------------------------
# Plot style
# ------------------------------------------------------------
sns.set_style("whitegrid")

fig, ax = plt.subplots(figsize=(6.8, 5.2))

bars = ax.bar(
    plot_prev["Phenotype"],
    plot_prev["Weighted_Percent"],
    width=0.62,
    edgecolor="black",
    linewidth=0.8
)

# ------------------------------------------------------------
# Add labels above bars
# ------------------------------------------------------------
for bar, value in zip(bars, plot_prev["Weighted_Percent"]):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.7,
        f"{value:.1f}%",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

# ------------------------------------------------------------
# Formatting
# ------------------------------------------------------------
ax.set_ylabel("Population-weighted prevalence (%)", fontsize=12)
ax.set_xlabel("Phenotype", fontsize=12)

ax.set_title(
    "Population-Representative Phenotype Prevalence",
    fontsize=13,
    pad=12
)

ax.set_ylim(0, 60)

# Remove top/right spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Improve tick appearance
ax.tick_params(axis='x', labelsize=11)
ax.tick_params(axis='y', labelsize=10)

# Light horizontal grid only
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.grid(axis="x", visible=False)

plt.tight_layout()

# ------------------------------------------------------------
# Save figures
# ------------------------------------------------------------
plt.savefig(
    os.path.join(FIG_DIR, "cluster_prevalence_weighted.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    os.path.join(FIG_DIR, "cluster_prevalence_weighted.pdf"),
    bbox_inches="tight"
)

plt.show()

In [ ]:
# %%
plt.figure(figsize=(6,5))
sns.barplot(x=prev_table["lab"].astype(str), y=prev_table["weighted_share"]*100)
plt.ylabel("Weighted Share (%)")
plt.xlabel("Cluster")
plt.title("Population-Weighted Cluster Prevalence")
for i, v in enumerate(prev_table["weighted_share"]*100):
    plt.text(i, v + 0.5, f"{v:.1f}%", ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "cluster_prevalence_weighted.png"), dpi=300)
plt.show()

# %% [markdown]
# ## 10) FIGURE — Weighted Phenotype Bar Plot (Cluster Means)

In [ ]:
# %%
viz_features = ["Age","BMI","Waist_Circumference","Systolic_BP","Diastolic_BP",
                "HbA1c","Fasting_Glucose","HDL_Cholesterol","LDL_Cholesterol"]
plot_data = summary_table[["Cluster"] + [f"{f}_mean" for f in viz_features]].copy()
plot_data.columns = ["Cluster"] + viz_features
melted = plot_data.melt(id_vars="Cluster", var_name="Feature", value_name="Weighted Mean")

plt.figure(figsize=(14,6))
sns.barplot(data=melted, x="Feature", y="Weighted Mean", hue="Cluster")
plt.xticks(rotation=45, ha='right')
plt.title("Population-Weighted Cluster Phenotypes (Means)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "cluster_weighted_means_barplot.png"), dpi=300)
plt.show()


In [ ]:
# %% [markdown]
# ## 11) Logistic Regression: Survey-Weighted (Obesity, Hypertension, Diabetes)
# Uses GLM Binomial with freq_weights as an approximation for complex survey weighting.


In [ ]:
# %%
import statsmodels.api as sm
import pandas as pd
import numpy as np
import os

reg_df = out.copy()

# ---- 1. Define outcomes ----
reg_df["obese"] = (reg_df["BMI"] >= 30).astype(int)
reg_df["htn"]   = ((reg_df["Systolic_BP"] >= 130) | (reg_df["Diastolic_BP"] >= 80)).astype(int)
reg_df["dm"]    = ((reg_df["Fasting_Glucose"] >= 126) | (reg_df["HbA1c"] >= 6.5)).astype(int)

# ---- 2. Choose which cluster labels to use (weighted consensus) ----
cluster_col = "cluster_weighted"   # or "cluster_unweighted" if you prefer

if cluster_col not in reg_df.columns:
    raise KeyError(f"Expected column '{cluster_col}' not found in reg_df. "
                   f"Available columns: {reg_df.columns.tolist()}")

Kcons = reg_df[cluster_col].nunique()

if Kcons == 2:
    # Binary indicator for cluster 1 (reference = cluster 0)
    reg_df["cluster1"] = (reg_df[cluster_col] == 1).astype(int)
    X_base = sm.add_constant(reg_df[["cluster1"]], has_constant="add")
else:
    # One-hot encode if more than 2 clusters
    X_base = pd.get_dummies(reg_df[cluster_col], prefix="cl", drop_first=True)
    X_base = sm.add_constant(X_base, has_constant="add")

# Keep alignment
X_base = X_base.loc[reg_df.index]

# ---- 3. Survey weights ----
# Make sure this matches your actual weight column name
weight_col = "Survey_Weight"   # change if needed (e.g. "WTMEC4YR", etc.)
if weight_col not in reg_df.columns:
    raise KeyError(f"Expected weight column '{weight_col}' not found. "
                   f"Available columns: {reg_df.columns.tolist()}")

weights_sm = reg_df[weight_col].astype(float).values

# ---- 4. Helper to fit GLM ----
def fit_glm_binom(y_col: str, X: pd.DataFrame):
    """Fit survey-weighted GLM Binomial with freq_weights; return tidy OR table and result."""
    y = reg_df[y_col].astype(int).values
    model = sm.GLM(y, X, family=sm.families.Binomial(), freq_weights=weights_sm)
    res = model.fit()

    params = res.params
    conf = res.conf_int()  # columns 0=lower, 1=upper

    rows = []
    for name in X.columns:
        if name == "const":
            continue
        OR  = float(np.exp(params[name]))
        L   = float(np.exp(conf.loc[name, 0]))
        U   = float(np.exp(conf.loc[name, 1]))
        pval = float(res.pvalues[name])
        rows.append(
            {"Predictor": name, "OR": OR, "CI_low": L, "CI_high": U, "p": pval}
        )
    return pd.DataFrame(rows), res

# ---- 5. Fit models ----
or_obese, res_obese = fit_glm_binom("obese", X_base)
or_htn,   res_htn   = fit_glm_binom("htn",   X_base)
or_dm,    res_dm    = fit_glm_binom("dm",    X_base)

# ---- 6. Combine tables ----
reg_table = pd.concat(
    [
        or_obese.assign(Outcome="Obesity"),
        or_htn.assign(Outcome="Hypertension"),
        or_dm.assign(Outcome="Diabetes"),
    ],
    ignore_index=True
)

# ---- 7. Round and save ----
reg_table_round = reg_table.copy()
reg_table_round[["OR", "CI_low", "CI_high"]] = reg_table_round[["OR", "CI_low", "CI_high"]].round(2)
reg_table_round["p"] = reg_table_round["p"].map(lambda x: f"{x:.3g}")
reg_table_round = reg_table_round[["Outcome", "Predictor", "OR", "CI_low", "CI_high", "p"]]

os.makedirs(OUT_DIR, exist_ok=True)
reg_table_round.to_csv(
    os.path.join(OUT_DIR, "logistic_regression_OR_table.csv"),
    index=False
)

print("\n[RESULT] Logistic regression (survey-weighted via GLM freq_weights):")
print(reg_table_round.to_string(index=False))


# %% [markdown]
# ## 12) FIGURE — Forest Plot of Odds Ratios (Cluster Effect)

In [ ]:
# %%
# Expect one predictor per outcome if K=2 ("cluster1")
plt.figure(figsize=(7,5))
plot_df = reg_table.copy()
plot_df["OR_label"] = plot_df.apply(lambda r: f"{r['OR']:.2f} [{r['CI_low']:.2f}, {r['CI_high']:.2f}]", axis=1)

ypos = np.arange(len(plot_df))[::-1]
plt.errorbar(plot_df["OR"], ypos, xerr=[plot_df["OR"]-plot_df["CI_low"], plot_df["CI_high"]-plot_df["OR"]],
             fmt='o', capsize=4)
plt.axvline(1.0, color="gray", linestyle="--")
plt.yticks(ypos, plot_df["Outcome"] + " — " + plot_df["Predictor"])
plt.xlabel("Odds Ratio (log scale)")
plt.xscale("log")
plt.title("Cluster Association with Outcomes (Survey-Weighted GLM)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "forest_OR_clusters.png"), dpi=300)
plt.show()

In [ ]:
# %%
import matplotlib.pyplot as plt
import numpy as np

# Ensure we’re using the regression table with OR and CI
plot_df = reg_table.copy()
plot_df["OR_label"] = plot_df.apply(
    lambda r: f"{r['OR']:.2f} [{r['CI_low']:.2f}, {r['CI_high']:.2f}]",
    axis=1
)

# Y positions (reverse order so first outcome is at top)
ypos = np.arange(len(plot_df))[::-1]

plt.figure(figsize=(8, 5))

# Error bars with points
plt.errorbar(
    plot_df["OR"], ypos,
    xerr=[plot_df["OR"] - plot_df["CI_low"], plot_df["CI_high"] - plot_df["OR"]],
    fmt='o', color='black', capsize=4, markersize=6
)

# Reference line at OR = 1
plt.axvline(1.0, color="gray", linestyle="--", linewidth=1)

# Customize y-axis labels
plt.yticks(ypos, plot_df["Outcome"] + " — " + plot_df["Predictor"], fontsize=11)

# Log-scale x-axis with clear, evenly spaced ticks
plt.xscale("log")
plt.xticks([0.5, 1, 2, 5], labels=["0.5", "1", "2", "5"], fontsize=11)

plt.xlabel("Odds Ratio (log scale)", fontsize=12)
plt.title("Cluster Association with Outcomes (Survey-Weighted GLM)", fontsize=13)

# Add grid for readability
plt.grid(axis="x", linestyle=":", color="lightgray")

# Add text labels for OR and CI next to each point
for i, (or_val, label) in enumerate(zip(plot_df["OR"], plot_df["OR_label"])):
    plt.text(or_val * 1.05, ypos[i], label, va="center", fontsize=10, color="black")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "forest_OR_clusters_clean.png"), dpi=300)
plt.show()




In [ ]:
# %%
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import os

# Ensure we’re using the regression table with OR and CI
plot_df = reg_table.copy()

# Build pretty text label: "1.53 [1.40, 1.68]"
plot_df["OR_label"] = plot_df.apply(
    lambda r: f"{r['OR']:.2f} [{r['CI_low']:.2f}, {r['CI_high']:.2f}]",
    axis=1
)

# Order outcomes (optional: ensure consistent ordering)
plot_df = plot_df.sort_values(by="Outcome", ascending=True).reset_index(drop=True)

# Y positions for plotting (top to bottom)
ypos = np.arange(len(plot_df))[::-1]

# Create figure
fig, ax = plt.subplots(figsize=(8, 5))

# Compute asymmetric error bars
err_low  = plot_df["OR"] - plot_df["CI_low"]
err_high = plot_df["CI_high"] - plot_df["OR"]
errors = [err_low, err_high]

# Draw points + CI whiskers
ax.errorbar(
    plot_df["OR"], ypos,
    xerr=errors,
    fmt='o',
    color='black',
    markersize=6,
    capsize=4,
    linewidth=1.4
)

# Reference line (OR = 1)
ax.axvline(1.0, color="gray", linestyle="--", linewidth=1)

# ---- FIXED LINE: cast to string to resolve TypeError ----
ax.set_yticks(ypos)
ax.set_yticklabels(
    plot_df["Outcome"].astype(str) + " — " + plot_df["Predictor"].astype(str),
    fontsize=11
)

# Log scale with nice ticks
ax.set_xscale("log")
ax.set_xticks([0.5, 1, 2, 5])
ax.set_xticklabels(["0.5", "1", "2", "5"], fontsize=11)

ax.set_xlabel("Odds Ratio (log scale)", fontsize=12)
ax.set_title("Cluster Association with Health Outcomes (Survey-Weighted GLM)", fontsize=13)

# Light grid for readability
ax.grid(axis="x", linestyle=":", color="lightgray")

# Add OR and CI text labels next to each point
for i, (or_val, label) in enumerate(zip(plot_df["OR"], plot_df["OR_label"])):
    ax.text(
        or_val * 1.05,
        ypos[i],
        label,
        va="center",
        fontsize=10,
        color="black"
    )

plt.tight_layout()

# Save the improved forest plot
plt.savefig(os.path.join(FIG_DIR, "forest_OR_clusters_clean.png"), dpi=300)
plt.show()


In [ ]:
# %%
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, FixedLocator

# ------------------------------------------------------------
# Forest plot: fixed ticks + plain labels + disable sci-notation
# ------------------------------------------------------------
# Assumes you already have:
#   - reg_table (DataFrame) with columns:
#       Outcome, Predictor, OR, CI_low, CI_high
#   - FIG_DIR (str) output folder (optional; will default to current dir)
# ------------------------------------------------------------

# Safety defaults
FIG_DIR = globals().get("FIG_DIR", os.getcwd())
os.makedirs(FIG_DIR, exist_ok=True)

# Copy + validate required columns
plot_df = reg_table.copy()
required = {"Outcome", "Predictor", "OR", "CI_low", "CI_high"}
missing = required - set(plot_df.columns)
if missing:
    raise KeyError(f"reg_table is missing required columns: {sorted(missing)}")

# Ensure numeric
for c in ["OR", "CI_low", "CI_high"]:
    plot_df[c] = pd.to_numeric(plot_df[c], errors="coerce")

# Drop invalid rows (optional but safer)
plot_df = plot_df.dropna(subset=["OR", "CI_low", "CI_high"]).reset_index(drop=True)

# Build label text: "1.53 [1.40, 1.68]"
plot_df["OR_label"] = plot_df.apply(
    lambda r: f"{r['OR']:.2f} [{r['CI_low']:.2f}, {r['CI_high']:.2f}]",
    axis=1
)

# Optional ordering (keeps plots stable across runs)
plot_df = plot_df.sort_values(by=["Outcome", "Predictor"], ascending=[True, True]).reset_index(drop=True)

# y positions top-to-bottom
ypos = np.arange(len(plot_df))[::-1]

# Asymmetric error bars
err_low = plot_df["OR"] - plot_df["CI_low"]
err_high = plot_df["CI_high"] - plot_df["OR"]
errors = [err_low, err_high]

# Create figure
fig, ax = plt.subplots(figsize=(9, 5.5))

# Points + CI whiskers
ax.errorbar(
    plot_df["OR"], ypos,
    xerr=errors,
    fmt="o",
    color="black",
    markersize=6,
    capsize=4,
    linewidth=1.4
)

# Reference line at OR = 1
ax.axvline(1.0, color="gray", linestyle="--", linewidth=1)

# Y labels
ax.set_yticks(ypos)
ax.set_yticklabels(
    plot_df["Outcome"].astype(str) + " — " + plot_df["Predictor"].astype(str),
    fontsize=11
)

# -----------------------------
# X axis: log scale + fixed ticks + plain labels (NO scientific notation)
# -----------------------------
ax.set_xscale("log")

# Choose ticks common for OR plots
ticks = [0.5, 1, 2, 5]
ax.xaxis.set_major_locator(FixedLocator(ticks))

# Force plain numeric formatting (no 1e0, no ×10^0)
fmt = ScalarFormatter()
fmt.set_scientific(False)
fmt.set_useOffset(False)
ax.xaxis.set_major_formatter(fmt)

# Also explicitly set tick labels as plain text (extra-safe)
ax.set_xticks(ticks)
ax.set_xticklabels([str(t) for t in ticks], fontsize=11)

ax.set_xlabel("Odds Ratio", fontsize=12)
ax.set_title("Cluster Association with Health Outcomes (Survey-Weighted GLM)", fontsize=13)

# Light grid (x only)
ax.grid(axis="x", linestyle=":", color="lightgray")

# Add OR text labels beside each point
# Place slightly to the right; for OR<1 we still place to the right of the point
for i, (or_val, label) in enumerate(zip(plot_df["OR"], plot_df["OR_label"])):
    ax.text(
        or_val * 1.06,   # small multiplicative offset works well on log-scale
        ypos[i],
        label,
        va="center",
        fontsize=10,
        color="black"
    )

plt.tight_layout()

out_path = os.path.join(FIG_DIR, "forest_OR_clusters_plain_ticks.png")
plt.savefig(out_path, dpi=300)
plt.show()

print(f"Saved forest plot to: {out_path}")


In [ ]:
# %% [markdown]
# ## 13) (Optional) Publication Table Export for Headline Variables

# %%
key_feats = ["BMI","Systolic_BP","Diastolic_BP","HbA1c","Fasting_Glucose","HDL_Cholesterol","LDL_Cholesterol"]
disp = summary_table[["Cluster"] + [f"{f}_mean" for f in key_feats]].copy()
disp.columns = ["Cluster"] + key_feats
disp_rounded = disp.copy()
for c in key_feats:
    disp_rounded[c] = disp_rounded[c].round(2)

disp_rounded.to_csv(os.path.join(OUT_DIR, "Cluster_Summary_Table.csv"), index=False)
disp_rounded.to_excel(os.path.join(OUT_DIR, "Cluster_Summary_Table.xlsx"), index=False)
print("\n[RESULT] Exported publication table (means only):")
print(disp_rounded.to_string(index=False))

In [ ]:
# %% Define helper before calling it
import os

def df_to_latex(df, path, caption, label, float_fmt="%.2f", index=False):
    """
    Save a pandas DataFrame as a LaTeX table (.tex).
    """
    tex = df.to_latex(index=index,
                      float_format=(lambda x: float_fmt % x),
                      caption=caption,
                      label=label,
                      escape=False)
    with open(path, "w", encoding="utf-8") as f:
        f.write(tex)
    print(f"[SAVED] {path}")

# %% Then create Table 1
cohort_cols = ["Age","Sex","BMI","Waist_Circumference","Systolic_BP","Diastolic_BP",
               "HbA1c","Fasting_Glucose","HDL_Cholesterol","LDL_Cholesterol"]

t1 = out[cohort_cols].describe().T[["mean","std","min","25%","50%","75%","max"]].rename(
    columns={"mean":"Mean","std":"SD","min":"Min","50%":"Median","75%":"P75","25%":"P25","max":"Max"}
)

OUT_DIR = r"C:\Users\afees\NHANES_OUT"   # <- update if needed
df_to_latex(
    t1.round(2),
    os.path.join(OUT_DIR, "tab1_baseline.tex"),
    caption="Baseline characteristics of the analytic cohort (NHANES 2015–2018).",
    label="tab:baseline"
)


In [ ]:
# %%
cohort_cols = ["Age","Sex","BMI","Waist_Circumference","Systolic_BP","Diastolic_BP",
               "HbA1c","Fasting_Glucose","HDL_Cholesterol","LDL_Cholesterol"]
t1 = out[cohort_cols].describe().T[["mean","std","min","25%","50%","75%","max"]].rename(
    columns={"mean":"Mean","std":"SD","min":"Min","50%":"Median","75%":"P75","25%":"P25","max":"Max"}
)
# Save LaTeX
df_to_latex(
    t1.round(2),
    os.path.join(OUT_DIR, "tab1_baseline.tex"),
    caption="Baseline characteristics of the analytic cohort (NHANES 2015–2018).",
    label="tab:baseline"
)


In [ ]:
!pip install Jinja2

In [ ]:
# %% Weighted vs Unweighted: prevalence comparison + ARI + (optional) agreement heatmap
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import adjusted_rand_score

FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# -------- Helpers
def weighted_share(labels, weights):
    dfp = pd.DataFrame({"lab": labels, "w": weights})
    g = dfp.groupby("lab", as_index=False)["w"].sum().sort_values("lab")
    g["weighted_share"] = g["w"] / g["w"].sum()
    return g[["lab", "weighted_share"]]

def sample_share(labels):
    s = pd.Series(labels).value_counts(normalize=True).sort_index()
    return pd.DataFrame({"lab": s.index, "sample_share": s.values})

# -------- Build prevalence tables safely
missing = []
try:
    labs_w = np.asarray(consensus_labels_weighted)
except NameError:
    missing.append("consensus_labels_weighted")

try:
    labs_u = np.asarray(consensus_labels_unweighted)
except NameError:
    missing.append("consensus_labels_unweighted")

try:
    weights_vec = np.asarray(w, dtype=float)
except NameError:
    missing.append("w (survey weights)")

if missing:
    print(f"[WARN] Missing objects: {', '.join(missing)}. Some plots may be skipped.")

# Weighted prevalence (if possible)
prev_w = None
if "consensus_labels_weighted" not in missing and "w (survey weights)" not in missing:
    prev_w = weighted_share(labs_w, weights_vec).rename(columns={"weighted_share":"share_weighted"})
    prev_w["share_weighted_pct"] = prev_w["share_weighted"] * 100
    print("[INFO] Weighted prevalence:\n", prev_w)

# Unweighted prevalence (sample proportions) — for weighted labels
prev_s_weighted = None
if "consensus_labels_weighted" not in missing:
    prev_s_weighted = sample_share(labs_w).rename(columns={"sample_share":"share_sample_weighted"})
    prev_s_weighted["share_sample_weighted_pct"] = prev_s_weighted["share_sample_weighted"] * 100

# Unweighted prevalence (sample proportions) — for unweighted labels
prev_unw = None
if "consensus_labels_unweighted" not in missing:
    prev_unw = sample_share(labs_u).rename(columns={"sample_share":"share_sample_unweighted"})
    prev_unw["share_sample_unweighted_pct"] = prev_unw["share_sample_unweighted"] * 100

# -------- Merge a tidy comparison table when possible
compare_df = None
if prev_w is not None and prev_unw is not None:
    compare_df = prev_w.merge(prev_unw, on="lab", how="outer").sort_values("lab")
elif prev_s_weighted is not None and prev_unw is not None:
    compare_df = prev_s_weighted.merge(prev_unw, on="lab", how="outer").sort_values("lab")

if compare_df is not None:
    print("\n[INFO] Prevalence comparison table:\n", compare_df)

# -------- Compute ARI if both label arrays exist and same K
ARI_txt = "NA"
if ("consensus_labels_weighted" not in missing) and ("consensus_labels_unweighted" not in missing):
    k_w = len(np.unique(labs_w))
    k_u = len(np.unique(labs_u))
    if k_w == k_u:
        ari_val = adjusted_rand_score(labs_w, labs_u)
        ARI_txt = f"{ari_val:.3f}"
        print(f"[RESULT] Adjusted Rand Index (weighted vs unweighted): {ARI_txt}")
    else:
        print(f"[NOTE] ARI skipped: K differs (weighted K={k_w}, unweighted K={k_u}).")

# -------- Figure: side-by-side prevalence bars
plt.figure(figsize=(8, 5))

# X locations based on union of cluster labels (so bars align)
if compare_df is not None:
    labs_sorted = compare_df["lab"].astype(int).tolist()
else:
    # fallback if compare_df missing
    labs_sorted = sorted(list(set(
        ([] if "consensus_labels_weighted" in missing else np.unique(labs_w)) |
        ([] if "consensus_labels_unweighted" in missing else np.unique(labs_u))
    )))

x = np.arange(len(labs_sorted))
bar_w = 0.35

# Bars:
# left = weighted %, right = unweighted % (sample) for the unweighted solution
left_vals = []
right_vals = []

if prev_w is not None:
    w_map = dict(zip(prev_w["lab"].astype(int), prev_w["share_weighted_pct"]))
    left_vals = [w_map.get(k, np.nan) for k in labs_sorted]
    plt.bar(x - bar_w/2, left_vals, width=bar_w, label="Weighted (population %)")
else:
    # fallback: use sample % of weighted labels if available
    if prev_s_weighted is not None:
        sw_map = dict(zip(prev_s_weighted["lab"].astype(int),
                          prev_s_weighted["share_sample_weighted_pct"]))
        left_vals = [sw_map.get(k, np.nan) for k in labs_sorted]
        plt.bar(x - bar_w/2, left_vals, width=bar_w, label="Weighted labels (sample %)")

if prev_unw is not None:
    u_map = dict(zip(prev_unw["lab"].astype(int), prev_unw["share_sample_unweighted_pct"]))
    right_vals = [u_map.get(k, np.nan) for k in labs_sorted]
    plt.bar(x + bar_w/2, right_vals, width=bar_w, label="Unweighted (sample %)")

# Labels & cosmetics
plt.xticks(x, [f"{k}" for k in labs_sorted])
plt.ylabel("Share (%)")
plt.xlabel("Cluster")
plt.title("Weighted vs Unweighted Consensus — Prevalence Comparison")
plt.grid(axis="y", linestyle=":", color="lightgray")

# Annotate ARI if available
if ARI_txt != "NA":
    plt.text(0.98, 0.95, f"ARI = {ARI_txt}",
             ha="right", va="top", transform=plt.gca().transAxes,
             fontsize=11, bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"))

plt.legend()
plt.tight_layout()
out_path = os.path.join(FIG_DIR, "fig6_weighted_vs_unweighted_prevalence.png")
plt.savefig(out_path, dpi=300)
plt.show()
print(f"[SAVED] {out_path}")

# -------- OPTIONAL: Agreement heatmap (confusion matrix) if same K
try:
    import seaborn as sns
    if ("consensus_labels_weighted" not in missing) and ("consensus_labels_unweighted" not in missing) and (len(np.unique(labs_w)) == len(np.unique(labs_u))):
        cm = pd.crosstab(pd.Series(labs_w, name="Weighted"), pd.Series(labs_u, name="Unweighted"))
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.title("Cluster Agreement (Counts)")
        plt.tight_layout()
        out_path2 = os.path.join(FIG_DIR, "fig6b_weighted_vs_unweighted_agreement.png")
        plt.savefig(out_path2, dpi=300)
        plt.show()
        print(f"[SAVED] {out_path2}")
except Exception as e:
    print(f"[NOTE] Skipped heatmap: {e}")


In [ ]:
# ============================================================
# Ablation / Comparison Study
# Weighted Ensemble vs Unweighted Ensemble vs Weighted K-means vs GMM
# ============================================================

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

K_SELECTED = 2

# ------------------------------------------------------------
# Comparator 1: Survey-weighted ensemble
# Already computed earlier
# ------------------------------------------------------------
labels_weighted_ensemble = consensus_labels_weighted

# ------------------------------------------------------------
# Comparator 2: Unweighted ensemble
# Already computed earlier
# ------------------------------------------------------------
labels_unweighted_ensemble = consensus_labels_unweighted

# ------------------------------------------------------------
# Comparator 3: Weighted k-means alone
# ------------------------------------------------------------
labels_weighted_kmeans = weighted_kmeans(
    X,
    w,
    n_clusters=K_SELECTED,
    seed=42
)

# ------------------------------------------------------------
# Comparator 4: GMM alone
# ------------------------------------------------------------
gmm_model = GaussianMixture(
    n_components=K_SELECTED,
    covariance_type="full",
    random_state=42,
    n_init=10
)

labels_gmm = gmm_model.fit_predict(X)


# ------------------------------------------------------------
# Helper: weighted prevalence
# ------------------------------------------------------------
def get_weighted_prevalence(labels, weights):
    df_prev = pd.DataFrame({
        "Cluster": labels,
        "Weight": weights
    })
    prev = df_prev.groupby("Cluster")["Weight"].sum()
    prev = prev / prev.sum()
    return prev.sort_index()


# ------------------------------------------------------------
# Helper: evaluation function
# ------------------------------------------------------------
def evaluate_solution(method_name, labels, X, weights, reference_labels):
    labels = np.asarray(labels)

    row = {
        "Method": method_name,
        "K": len(np.unique(labels)),
        "ARI_vs_weighted_ensemble": adjusted_rand_score(reference_labels, labels),
        "NMI_vs_weighted_ensemble": normalized_mutual_info_score(reference_labels, labels),
        "Silhouette": silhouette_score(X, labels),
        "Davies_Bouldin": davies_bouldin_score(X, labels),
        "Calinski_Harabasz": calinski_harabasz_score(X, labels)
    }

    prev = get_weighted_prevalence(labels, weights)

    for cluster_id, share in prev.items():
        row[f"Weighted_prev_cluster_{cluster_id}"] = share * 100

    return row


# ------------------------------------------------------------
# Build ablation table
# ------------------------------------------------------------
ablation_rows = []

ablation_rows.append(
    evaluate_solution(
        "Survey-weighted ensemble",
        labels_weighted_ensemble,
        X,
        w,
        labels_weighted_ensemble
    )
)

ablation_rows.append(
    evaluate_solution(
        "Unweighted ensemble",
        labels_unweighted_ensemble,
        X,
        w,
        labels_weighted_ensemble
    )
)

ablation_rows.append(
    evaluate_solution(
        "Weighted k-means",
        labels_weighted_kmeans,
        X,
        w,
        labels_weighted_ensemble
    )
)

ablation_rows.append(
    evaluate_solution(
        "Gaussian mixture model",
        labels_gmm,
        X,
        w,
        labels_weighted_ensemble
    )
)

ablation_table = pd.DataFrame(ablation_rows)

# Round numeric columns
num_cols = ablation_table.select_dtypes(include=[np.number]).columns
ablation_table[num_cols] = ablation_table[num_cols].round(3)

print("\n[RESULT] Ablation comparison table:")
print(ablation_table.to_string(index=False))

# Save CSV and LaTeX versions
ablation_table.to_csv(
    os.path.join(OUT_DIR, "ablation_comparison_table.csv"),
    index=False
)

ablation_table.to_latex(
    os.path.join(OUT_DIR, "ablation_comparison_table.tex"),
    index=False,
    escape=False,
    caption="Ablation comparison of the survey-weighted ensemble against simpler clustering alternatives.",
    label="tab:ablation_comparison"
)